# AIaaS — Portable LoRA / QLoRA Fine-Tune Benchmark

A cross-platform **training** companion to the vLLM serving benchmark. It runs a
short, fixed-budget supervised fine-tune (SFT) of an ungated **Qwen2.5** model
with **LoRA / QLoRA** and reports the numbers that size a training box:

- **train tokens/s** and **samples/s** (throughput)
- **peak VRAM** (fit / headroom)
- **train wall-time** for a fixed step budget (comparability)
- **trainable-parameter %** (LoRA footprint)

### How it stays comparable across platforms
- **Fixed step budget** (`MAX_STEPS`) and **fixed sequence length** with packing,
  so wall-time and tokens/s line up between a Colab T4 and an A100.
- **VRAM-tiered, ungated models** (Qwen2.5-1.5B anchor / 7B large) — no HF token.
- **QLoRA (4-bit) is the portable anchor** that fits a T4; LoRA (16-bit) is used
  where VRAM is ample. Set `METHOD="qlora"` everywhere for a strict apples-to-apples
  run.

### Requirements
- A **CUDA GPU** runtime (A100 / Colab T4·L4·A100 / …). Will not run on CPU.
- ~5–15 min including model download.

> Targets recent `trl` (SFTConfig / `processing_class`). If a torch/CUDA lib is
> upgraded during install, **restart the runtime** and re-run from the install cell.

## 1. Install the fine-tuning stack

In [ ]:
import subprocess, sys

# trl -> SFTTrainer; peft -> LoRA; bitsandbytes -> 4-bit QLoRA.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "peft", "trl", "datasets", "accelerate",
                "bitsandbytes", "pandas"], check=False)

try:
    import transformers, peft, trl
    print("transformers", transformers.__version__,
          "| peft", peft.__version__, "| trl", trl.__version__)
except Exception as e:
    print("Imports not ready:", e)
    print("If torch/CUDA libs were upgraded, RESTART the runtime and re-run from this cell.")


## 2. Environment & hardware capture
Every result is tagged with this so on-prem vs Colab is comparable.

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
The small tier (≤16 GB) is the **apples-to-apples anchor** that runs on a T4
everywhere; the large tier uses bigger GPUs (your A100). Models are ungated
(Qwen2.5) so no HF token is needed.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
MODEL = "Qwen/Qwen2.5-7B-Instruct" if TIER == "large" else "Qwen/Qwen2.5-1.5B-Instruct"

# QLoRA (4-bit) is the portable anchor that fits a T4; LoRA (16-bit) where VRAM is
# ample. Force METHOD="qlora" everywhere for a strict cross-platform comparison.
METHOD = "lora" if VRAM >= 24 else "qlora"

# Pre-Ampere GPUs (e.g. Colab T4, compute capability 7.5) lack bf16; use fp16 there.
_cc_major = torch.cuda.get_device_capability(0)[0]
COMPUTE_DTYPE = "bfloat16" if _cc_major >= 8 else "float16"

DATASET = "yahma/alpaca-cleaned"   # ungated instruction-tuning set
MAX_SEQ_LEN = 1024
NUM_TRAIN_SAMPLES = 2000           # slice mapped before packing
MICRO_BATCH = 2 if TIER == "large" else 1
GRAD_ACCUM = 16
EFF_BATCH = MICRO_BATCH * GRAD_ACCUM
LR = 2e-4
MAX_STEPS = 60                     # fixed step budget -> comparable wall-time
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

OUT_DIR = "lora_train_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL={MODEL}, METHOD={METHOD}, "
      f"COMPUTE_DTYPE={COMPUTE_DTYPE}, EFF_BATCH={EFF_BATCH}")


## 4. Dataset
Map a fixed slice of Alpaca to chat-templated text. Packing (set later) then
concatenates these into uniform `MAX_SEQ_LEN` blocks so tokens/s is comparable.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

raw = load_dataset(DATASET, split="train").select(range(NUM_TRAIN_SAMPLES))

def to_text(ex):
    instr = (ex.get("instruction") or "").strip()
    inp = (ex.get("input") or "").strip()
    out = (ex.get("output") or "").strip()
    user = instr if not inp else f"{instr}\n\n{inp}"
    msgs = [{"role": "user", "content": user},
            {"role": "assistant", "content": out}]
    return {"text": tok.apply_chat_template(msgs, tokenize=False)}

train_ds = raw.map(to_text, remove_columns=raw.column_names)
print(train_ds)
print("---- sample ----")
print(train_ds[0]["text"][:300])


## 5. Load model + attach LoRA
For QLoRA the base weights are loaded in 4-bit (NF4) and frozen; only the LoRA
adapters train. For LoRA the base loads in `COMPUTE_DTYPE`.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

dtype = torch.bfloat16 if COMPUTE_DTYPE == "bfloat16" else torch.float16

quant_cfg = None
if METHOD == "qlora":
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quant_cfg,
    torch_dtype=dtype,
    device_map="auto",
)
model.config.use_cache = False
if METHOD == "qlora":
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
# Needed for gradient checkpointing to backprop into the (frozen) inputs.
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")


## 6. Train (fixed step budget) and capture metrics

In [ ]:
import time, dataclasses
from trl import SFTTrainer, SFTConfig

torch.cuda.reset_peak_memory_stats()

sft_kwargs = dict(
    output_dir=os.path.join(OUT_DIR, "ckpt"),
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit" if METHOD == "qlora" else "adamw_torch",
    bf16=(COMPUTE_DTYPE == "bfloat16"),
    fp16=(COMPUTE_DTYPE == "float16"),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    packing=True,
    dataset_text_field="text",
    report_to="none",
    save_strategy="no",
)
# trl renamed the packing/truncation length field max_seq_length -> max_length;
# set whichever the installed version exposes so a fresh -U install doesn't error.
_sft_fields = {f.name for f in dataclasses.fields(SFTConfig)}
sft_kwargs["max_length" if "max_length" in _sft_fields else "max_seq_length"] = MAX_SEQ_LEN
sft_cfg = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    processing_class=tok,
)

t0 = time.time()
train_out = trainer.train()
elapsed = time.time() - t0

m = train_out.metrics
runtime = m.get("train_runtime", elapsed)
peak_vram = torch.cuda.max_memory_allocated() / 1e9
# Packing makes every step consume EFF_BATCH * MAX_SEQ_LEN tokens.
tokens = MAX_STEPS * EFF_BATCH * MAX_SEQ_LEN

METRICS = {
    "train_runtime_s": round(runtime, 2),
    "train_samples_per_second": round(m.get("train_samples_per_second", 0.0), 3),
    "train_steps_per_second": round(m.get("train_steps_per_second", 0.0), 3),
    "train_tokens_per_second": round(tokens / runtime, 1) if runtime else None,
    "peak_vram_gb": round(peak_vram, 2),
    "final_train_loss": round(m.get("train_loss", float("nan")), 4),
    "trainable_params": int(trainable),
    "total_params": int(total),
    "trainable_pct": round(100 * trainable / total, 3),
}
print(json.dumps(METRICS, indent=2))


## 7. Results — normalize, save, summarize

In [ ]:
import pandas as pd

NORM = {
    "schema": "lora-train-bench/1.0",
    "env": ENV, "tier": TIER, "model": MODEL,
    "method": METHOD, "compute_dtype": COMPUTE_DTYPE,
    "dataset": DATASET, "max_seq_len": MAX_SEQ_LEN,
    "max_steps": MAX_STEPS, "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM, "effective_batch": EFF_BATCH,
    "metrics": METRICS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"lora_train_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)

print("Saved:", out)
print("\n==== TRAINING SUMMARY ====")
print(f"{ENV['gpu_name']} ({ENV['vram_total_gb']} GB) | {MODEL} | {METHOD} | {COMPUTE_DTYPE}")
df = pd.DataFrame([METRICS])
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string(index=False))


## 8. Teardown
Free the GPU before running anything else.

In [ ]:
import gc

for name in ("trainer", "model"):
    if name in dir():
        try:
            exec(f"del {name}")
        except Exception:
            pass
gc.collect()
torch.cuda.empty_cache()
print("Freed GPU memory.")


## 9. Reading the result (and what to compare)

- **train tokens/s** is your raw training throughput; on a fixed `MAX_STEPS`
  budget the **wall-time** is directly comparable between platforms.
- **peak VRAM** tells you the headroom — if QLoRA on the small tier already nears
  the T4's ceiling, that workload won't move to LoRA/16-bit there.
- Collect the per-platform JSONs and line them up with the serving numbers:
  ```bash
  python compare_results.py 'lora_train_results/*.json' 'vllm_bench_results/*.json'
  ```

For a strict cross-platform comparison, set `METHOD="qlora"` on every platform so
the only variable is the hardware. See `BENCHMARKING_STRATEGY.md` for the suite.